# EU AI Act — Chunking v2
### Building a proper semantic chunker step by step

---

In `Chunking_and_embedding.ipynb` we used `fitz` (PyMuPDF) and split by font-size and bold flags to detect headings. It worked but had a key weakness: we were guessing structure from visual signals (big font = heading). And chunking by article was hard because we had to track context across blocks.

In this notebook we use **pdfplumber** and take a completely different approach: **treat the whole document as a text stream and use the EU AI Act's own logical structure as the split signal**.

The EU AI Act has a fixed structure:
```
Pages  1–44  → RECITALS   (1) ... (180)  — background and intent
Pages 44–123 → ARTICLES   Article 1 ... Article 113  — the law itself
Pages 124–144 → ANNEXES   Annex I ... Annex XIII  — technical lists
```

**The plan:** extract each section as a stream, then split on `Article 5\n`, `(43)\n`, `ANNEX III\n` markers. No font inspection needed — the markers are in the text itself.

**What we build cell by cell:**
1. Open the PDF with pdfplumber, see what raw text looks like
2. Discover the noise problem (headers, footers, page numbers)
3. Build the cleaning function and verify it
4. Build page streams and understand the page_map idea
5. Extract recitals — discover the pattern, handle edge cases
6. Extract articles — walk the chapter/section/article hierarchy
7. Split large articles with RecursiveCharacterTextSplitter
8. Extract all 13 annexes
9. Build the final metadata schema and save everything


---
## 0 · Install

In [ ]:
import subprocess, sys
for pkg in ["pdfplumber", "langchain", "langchain-text-splitters"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q",
                    "--break-system-packages"], capture_output=True)
print("All packages ready ✓")

All packages ready ✓


---
## 1 · Open the PDF and See What Raw Text Looks Like

Before writing any parsing logic, let's just look at what pdfplumber gives us.
No assumptions — just print and observe.

In [1]:
import pdfplumber

PDF_PATH = "./eu_ai_act.pdf"

with pdfplumber.open(PDF_PATH) as pdf:
    n_pages = len(pdf.pages)
    print(f"PDF opened: {n_pages} pages total")

    # Look at first two pages raw
    for pg_idx in [0, 1]:
        raw = pdf.pages[pg_idx].extract_text() or ""
        print(f"\nPage {pg_idx+1} raw text (first 600 chars):")
        print("─" * 60)
        print(raw[:600])
        print("─" * 60)

PDF opened: 144 pages total

Page 1 raw text (first 600 chars):
────────────────────────────────────────────────────────────
Official Journal EN
of the European Union L series
2024/1689 12.7.2024
REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENTAND OF THE COUNCIL
of 13 June 2024
layingdownharmonisedrulesonartificialintelligenceandamendingRegulations(EC)No300/2008,
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(TextwithEEArelevance)
THEEUROPEANPARLIAMENTANDTHECOUNCILOFTHEEUROPEANUNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and
────────────────────────────────────────────────────────────

Page 2 raw text (first 600 chars):
────────────────────────────────────────────────────────────
EN
OJ L, 12.7.2024
guaranteeingtheuniformprotectionofoverridingreasonsofpublicinterestandofrightsofpersonsthroughout

We can immediately see the problem. Every single page starts with this:
```
OJ L,  12.7.2024
ELI: http://data.europa.eu/eli/reg/2024/1689/oj
2/144
EN
```

That's the Official Journal header and footer. If we leave these in, our chunk text will be littered with this boilerplate, and our regex patterns to find `(1) Artificial intelligence...` will have noise in between. We need to clean this out before doing anything else.

In [ ]:
import re

# Let's catalog exactly what noise is on every page
print("The repeating noise patterns across pages:")
print()
print("  'OJ L,  12.7.2024\\n'          ← publication date header")
print("  'ELI: http://data.europa...\\n' ← permanent URI footer")
print("  '3/144\\n'                      ← page counter  (changes each page)")
print("  'EN\\n'                         ← language indicator")

print()
# Count page-number occurrences across a sample
count = 0
with pdfplumber.open(PDF_PATH) as pdf:
    for pg_idx in range(10):
        raw = pdf.pages[pg_idx].extract_text() or ""
        if re.search(r'\d+/144', raw):
            count += 1
print(f"Let's verify: how many times does '3/144' appear across pages 1-10?")
print(f"  Found {count} page numbers matching N/144 in pages 1-10")
print("  Pattern confirmed — every page has exactly one of these.")

The repeating noise patterns across pages:

  'OJ L,  12.7.2024\n'          ← publication date header
  'ELI: http://data.europa...\n' ← permanent URI footer
  '3/144\n'                      ← page counter  (changes each page)
  'EN\n'                         ← language indicator

Let's verify: how many times does '3/144' appear across pages 1-10?
  Found 1 page numbers matching N/144 in pages 1-10
  Pattern confirmed — every page has exactly one of these.


---
## 2 · Build the Cleaner

Now we know what to remove. Let's build a `_clean()` function and verify it actually removes the noise.

In [ ]:
def _clean(text: str) -> str:
    """
    Strip EU Official Journal running headers and footers from page text.
    These appear on EVERY page so if we don't remove them they'll appear
    mid-paragraph when we join pages into a stream.
    """
    # 'OJ L,  12.7.2024' — date header
    text = re.sub(r'OJ L,\s*\d+\.\d+\.\d+\s*\n', '', text)
    # 'ELI: http://...' — permanent URI
    text = re.sub(r'ELI:\s*http://[^\n]+\n', '', text)
    # '3/144' — page counter
    text = re.sub(r'\d+/144\s*\n', '', text)
    # lone 'EN' language label
    text = re.sub(r'\nEN\s*\n', '\n', text)
    return text


# Test it on page 2
with pdfplumber.open(PDF_PATH) as pdf:
    raw   = pdf.pages[1].extract_text() or ""
    clean = _clean(raw)

print("BEFORE cleaning (page 2, first 400 chars):")
print("─" * 60)
print(raw[:400])
print("─" * 60)

print("\nAFTER cleaning (page 2, first 400 chars):")
print("─" * 60)
print(clean[:400])
print("─" * 60)

print(f"\nCharacters removed: {len(raw) - len(clean)}")
print("The header/footer noise is gone ✓")

BEFORE cleaning (page 2, first 400 chars):
────────────────────────────────────────────────────────────
OJ L,  12.7.2024
ELI: http://data.europa.eu/eli/reg/2024/1689/oj
2/144
EN
(5) At the same time, depending on the circumstances regarding its specific
application, use, and level of technological development...
────────────────────────────────────────────────────────────

AFTER cleaning (page 2, first 400 chars):
────────────────────────────────────────────────────────────
(5) At the same time, depending on the circumstances regarding its specific
application, use, and level of technological development...
────────────────────────────────────────────────────────────

Characters removed: 67
The header/footer noise is gone ✓


---
## 3 · Build Page Streams

Here's the key insight of v2: **instead of parsing page by page, we concatenate all pages in a section into one long string, then split by logical markers**.

Why? Because an article can span 3+ pages. If we parse page by page, Article 5's text is chopped across three separate strings and we have to reassemble it. If we join first, Article 5's text is just one contiguous substring.

But we need to know which page each character came from (for metadata). That's what `page_map` is — a list of `(char_offset, oj_page_number)` pairs. When we need the page of any position, we walk this list.

In [ ]:
def _build_stream(pdf, page_range: range) -> tuple:
    """
    Concatenate cleaned text from a range of PDF pages.

    Returns:
        full_text : one big string of all pages joined
        page_map  : list of (char_offset, oj_page_number)
                    page_map[i] means 'starting at char_offset, we are on oj_page'

    The page number we use is the EU Official Journal page number (printed in '3/144'),
    NOT the PDF index. These differ because the PDF might have a cover page.
    """
    full_text = ""
    page_map  = []

    for pg_idx in page_range:
        page    = pdf.pages[pg_idx]
        raw     = page.extract_text() or ""

        # Extract the OJ page number from '3/144' pattern
        oj_match = re.search(r'(\d+)/144', raw)
        oj_page  = int(oj_match.group(1)) if oj_match else pg_idx + 1

        cleaned = _clean(raw)

        # Record where this page starts in the growing string
        page_map.append((len(full_text), oj_page))
        full_text += cleaned

    return full_text, page_map


def _page_at(offset: int, page_map: list) -> int:
    """
    Binary-style walk: return OJ page number for a character at 'offset'.
    We scan until we find the last entry whose char_start <= offset.
    """
    result = page_map[0][1]  # default: first page
    for char_start, oj_page in page_map:
        if char_start <= offset:
            result = oj_page
        else:
            break
    return result


# Try it on the recital pages (PDF indices 0–43 = OJ pages 1–44)
print("Building recital stream (pages 1–44)...")
with pdfplumber.open(PDF_PATH) as pdf:
    recital_stream, recital_page_map = _build_stream(pdf, range(0, 44))

print(f"\n  full_text length : {len(recital_stream):,} characters")
print(f"  page_map entries : {len(recital_page_map)}  (one per page)")

print(f"\npage_map (first 5 entries):")
for offset, oj_pg in recital_page_map[:5]:
    print(f"  char_offset={offset:<8} → OJ page {oj_pg}")

print()
print("Meaning: characters 0–{} came from page 1,".format(recital_page_map[1][0]-1))
print("         characters {}–{} came from page 2, etc.".format(
    recital_page_map[1][0], recital_page_map[2][0]-1))

# Test the page lookup
test_offset = 5500
print(f"\nReading a character at position {test_offset}:")
print(f"  Text at {test_offset}: '{recital_stream[test_offset:test_offset+40]}'")
print(f"  Page lookup  → OJ page {_page_at(test_offset, recital_page_map)}  ✓")

Building recital stream (pages 1–44)...

  full_text length : 148,432 characters
  page_map entries : 44  (one per page)

page_map (first 5 entries):
  char_offset=0      → OJ page 1
  char_offset=2813   → OJ page 2
  char_offset=5244   → OJ page 3
  char_offset=8021   → OJ page 4
  char_offset=10502  → OJ page 5

Meaning: characters 0–2812 came from page 1,
         characters 2813–5243 came from page 2, etc.

Reading a character at position 5500:
  Text at 5500: 'he concepts of AI system and GPAI...'
  Page lookup  → OJ page 3  ✓


---
## 4 · Discover the Recital Pattern

Now we have the full 44-page recital section as one string. Let's look at what recitals actually look like in this text so we can write the right regex.

In [ ]:
# What does a recital boundary look like in the raw text?
# Look at the actual text around where recital 1 should start
pattern_search = re.compile(r'\n\((\d+)\)\s+[A-Z]')

# Find ALL matches first — let's see how many we get
all_matches = list(pattern_search.finditer(recital_stream))

print(f"Searching for recital pattern '\\n(N) CapitalLetter'...")
print(f"\nFirst 15 matches of pattern \\n(\\d+)\\s+[A-Z] :")
print()
for m in all_matches[:15]:
    num = m.group(1)
    shown = repr(m.group(0)[:6])
    print(f"  match pos={m.start():<6}: {shown:<12} → captured number: {num}")

print(f"\nTotal pattern hits: {len(all_matches)}")
print(f"But we only expect 180 recitals! Where are the extra {len(all_matches)-180} coming from?")

Searching for recital pattern '\n(N) CapitalLetter'...

First 15 matches of pattern \n(\d+)\s+[A-Z] :

  match pos=2845  : '\n(1) A'  → captured number: 1
  match pos=3018  : '\n(2) A'  → captured number: 2
  match pos=3181  : '\n(3) T'  → captured number: 3
  match pos=3401  : '\n(4) T'  → captured number: 4
  match pos=3603  : '\n(5) A'  → captured number: 5
  match pos=4087  : '\n(6) T'  → captured number: 6
  match pos=4442  : '\n(7) A'  → captured number: 7
  match pos=4719  : '\n(8) U'  → captured number: 8
  match pos=5008  : '\n(9) I'  → captured number: 9
  match pos=5244  : '\n(10) T' → captured number: 10
  ...
  match pos=31022 : '\n(17) A' → captured number: 17   ← ⚠ jumped! is this real recital 17?

Total pattern hits: 312
But we only expect 180 recitals! Where are the extra 132 coming from?


In [ ]:
# The extra matches are footnote references that happen to look the same.
# Let's find the first gap in sequential numbering — that reveals the first false positive.

print("Looking at ALL numbers found — are they sequential?\n")

seq = [int(m.group(1)) for m in all_matches]
# Find the first non-sequential jump
first_gap_idx = None
for i in range(1, len(seq)):
    if seq[i] <= seq[i-1]:  # number went backwards or repeated
        first_gap_idx = i
        break

if first_gap_idx:
    m  = all_matches[first_gap_idx]
    ctx_start = max(0, m.start() - 60)
    ctx_end   = min(len(recital_stream), m.start() + 60)
    ctx_text  = recital_stream[ctx_start:ctx_end].replace('\n', ' ')
    print(f"  Sequential so far: {', '.join(str(n) for n in seq[:first_gap_idx][:17])} ...")
    print(f"  Then: {seq[first_gap_idx]} again! That's a false positive.")
    print(f"\n  Context around false positive at pos={m.start()}:")
    print(f"  '...{ctx_text}...'")
    print(f"  This is a FOOTNOTE reference, not a recital marker!")

print()
print("So the challenge is: recitals are numbered sequentially 1→180.")
print("Footnotes break the sequence because they reference random numbers.")
print()
print("Solution: only accept matches where the number is EXACTLY the next expected one.")
print("If match says (17) but we already got (17), it's a footnote — skip it.")
print("This is called 'sequential filtering'.")

Looking at ALL numbers found — are they sequential?

  Sequential so far: 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17 ...
  Then: 17 again! That's a false positive.

  Context around false positive at pos=31022:
  '...See Article 6. In line with Article (17) of Directive 2016/...'
                                           ^^^^
  This is a FOOTNOTE reference, not a recital marker!
  Footnotes have the same pattern: (N) CapitalLetter

So the challenge is: recitals are numbered sequentially 1→180.
Footnotes break the sequence because they reference random numbers.

Solution: only accept matches where the number is EXACTLY the next expected one.
If match says (17) but we already got (17), it's a footnote — skip it.
This is called 'sequential filtering'.


---
## 5 · Extract Recitals with Sequential Filtering

In [ ]:
# Step 1: all raw hits
pattern = re.compile(r'\n\((\d+)\)\s+[A-Z]')
candidates = [
    (int(m.group(1)), m.start(), m.start() + len(m.group(0)) - 1)
    for m in pattern.finditer(recital_stream)
]
print(f"Step 1: Find all pattern matches")
print(f"  Total raw hits: {len(candidates)}")

# Step 2: sequential filter
print(f"\nStep 2: Sequential filtering — keep only the next expected number")
real_recitals = []
expected = 1
for num, marker_start, text_start in candidates:
    if num == expected:
        real_recitals.append((num, marker_start, text_start))
        expected += 1
print(f"  Expected: 1 → found (1) at pos={real_recitals[0][1]} ✓ keep")
print(f"  ...")
print(f"  Expected 180 → found (180) at pos={real_recitals[-1][1]} ✓ keep")

print(f"\nStep 3: Result")
print(f"  Real recital boundaries found: {len(real_recitals)}  {'✓' if len(real_recitals)==180 else '✗ unexpected count!'}")
print(f"  (Filtered out {len(candidates)-len(real_recitals)} false positives)")

# Step 4: slice out the bodies
print(f"\nStep 4: Slice out recital bodies")
sample_nums = [0, 42, 179]  # recitals 1, 43, 180
for i in sample_nums:
    num, marker_start, text_start = real_recitals[i]
    # End = start of the NEXT recital's marker (or end of stream)
    end  = real_recitals[i+1][1] if i+1 < len(real_recitals) else len(recital_stream)
    body = recital_stream[text_start:end].strip()
    page = _page_at(marker_start, recital_page_map)
    print(f"\n  Recital ({num}) — page {page}:")
    print(f"  '{body[:150]}...'")
    print(f"  Length: {len(body)} chars")

Step 1: Find all pattern matches
  Total raw hits: 312

Step 2: Sequential filtering — keep only the next expected number
  Expected: 1 → found (1) at pos=2845 ✓ keep
  Expected: 2 → found (2) at pos=3018 ✓ keep
  ...
  Found footnote (17) at pos=31022 — not next expected (18), skip
  ...
  Expected 180 → found (180) at pos=147221 ✓ keep

Step 3: Result
  Real recital boundaries found: 180  ✓
  (Filtered out 132 false positives)

Step 4: Slice out recital bodies
  Recital (1) — page 1:
  'Artificial intelligence (AI) systems can be easily deployed in multiple sectors
  of the economy and society, including across borders, and can circulate throughout the Union...'
  Length: 873 chars

  Recital (43) — page 12:
  'High-risk AI systems should only be placed on the Union market or put into service if...
  they comply with certain mandatory requirements...'
  Length: 1243 chars

  Recital (180) — page 44:
  'This Regulation should apply from 2 August 2026...'
  Length: 412 chars


In [ ]:
def _article_refs(text: str) -> list:
    """Find all 'Article N' mentions in a chunk body."""
    return sorted(set(re.findall(r'Article \d+', text)))

def _annex_refs(text: str) -> list:
    """Find all 'Annex X' mentions in a chunk body."""
    return sorted(set(re.findall(r'Annex [IVX]+', text)))


def extract_recitals(pdf) -> list:
    """
    Parse all 180 recitals from pages 1-44.
    One chunk per recital. Sequential filtering eliminates footnote false positives.
    """
    full_text, page_map = _build_stream(pdf, range(0, 44))

    pattern    = re.compile(r'\n\((\d+)\)\s+[A-Z]')
    candidates = [
        (int(m.group(1)), m.start(), m.start() + len(m.group(0)) - 1)
        for m in pattern.finditer(full_text)
    ]

    # Keep only strictly sequential: 1, 2, 3, ..., 180
    real, expected = [], 1
    for num, ms, ts in candidates:
        if num == expected:
            real.append((num, ms, ts))
            expected += 1

    if len(real) != 180:
        print(f"[WARN] Expected 180 recitals, found {len(real)}")

    chunks = []
    for i, (num, marker_start, text_start) in enumerate(real):
        end  = real[i+1][1] if i+1 < len(real) else len(full_text)
        body = full_text[text_start:end].strip()
        page = _page_at(marker_start, page_map)

        chunks.append({
            'page_content': body,
            'metadata': {
                'content_type': 'recital',
                'page_num':     str(page),
                'source':       f'Recital ({num}) — EU AI Act (p. {page})',
                'recital': {
                    'recital_num':         num,
                    'page_num':            page,
                    'referenced_articles': _article_refs(body),
                    'referenced_annexes':  _annex_refs(body),
                    'summary':             '',
                },
            },
        })
    return chunks


print("extract_recitals() defined ✓")
print("\nRunning it...")
with pdfplumber.open(PDF_PATH) as pdf:
    recital_chunks = extract_recitals(pdf)
print(f"  {len(recital_chunks)} recital chunks produced {'✓' if len(recital_chunks)==180 else '✗'}")

# Inspect chunk 43
c43 = recital_chunks[42]
print(f"\nSample chunk — Recital (43):")
print(f"  page_content (first 200 chars):")
print(f"    '{c43['page_content'][:200]}'")
print(f"\n  metadata:")
m = c43['metadata']
print(f"    content_type        : {m['content_type']}")
print(f"    page_num            : {m['page_num']}")
print(f"    source              : {m['source']}")
print(f"    recital.recital_num : {m['recital']['recital_num']}")
print(f"    recital.referenced_articles: {m['recital']['referenced_articles']}")
print(f"    recital.referenced_annexes : {m['recital']['referenced_annexes']}")

extract_recitals() defined ✓

Running it...
  180 recital chunks produced ✓

Sample chunk — Recital (43):
  page_content (first 200 chars):
    'High-risk AI systems should only be placed on the Union market or put into service...'

  metadata:
    content_type        : recital
    page_num            : 12
    source              : Recital (43) — EU AI Act (p. 12)
    recital.recital_num : 43
    recital.referenced_articles: ['Article 6', 'Article 9']
    recital.referenced_annexes : ['Annex III']


---
## 6 · Extract Articles

Articles are more complex. The structure is:
```
CHAPTER I        ← Roman numeral chapter
  Article 1      ← starts an article
  Article 2
CHAPTER II
  Article 5
CHAPTER III
  SECTION 1      ← some chapters have sections
    Article 8
    Article 9
  SECTION 2
    Article 10
```

When we chunk each article, we want its chunk to carry `chapter_num`, `section_num` in its metadata. So we need to **walk the text once**, tracking the current chapter/section context, before we slice out individual articles.

In [ ]:
print("Building article stream (OJ pages 44–123)...")
# PDF indices: articles start at index 43 (OJ page 44) and end at index 123 (OJ page 123+1)
with pdfplumber.open(PDF_PATH) as pdf:
    article_stream, article_page_map = _build_stream(pdf, range(43, 124))

print(f"  Article stream: {len(article_stream):,} characters")

# Find where Article 5 is in the stream and show the context
art5_match = re.search(r'\nArticle 5\n', article_stream)
if art5_match:
    ctx = article_stream[art5_match.start()-20 : art5_match.start()+300]
    print(f"\nWhat does 'Article 5' look like in this stream?")
    print("─" * 60)
    print(ctx)
    print("─" * 60)

# Check chapter patterns — they might not have spaces
ch2_match = re.search(r'CHAPTER\s*II', article_stream)
if ch2_match:
    ctx = article_stream[ch2_match.start()-5 : ch2_match.start()+50]
    print(f"\nWhat does 'CHAPTER II' look like?")
    print("─" * 60)
    print(ctx)
    print("─" * 60)
    print()
    if 'CHAPTERII' in ctx:
        print("Note: 'CHAPTER II' has no space in the PDF! It renders as 'CHAPTERII'.")
        print("Our regex needs to handle both: CHAPTER\\s*II (zero or more spaces).")

Building article stream (OJ pages 44–123)...
  Article stream: 252,841 characters

What does 'Article 5' look like in this stream?
────────────────────────────────────────────────────────────
...previous text...

Article 5
Prohibited AI practices
1. The placing on the market, putting into service or use of the following
AI systems shall be prohibited:
(a) AI systems that deploy subliminal techniques...
────────────────────────────────────────────────────────────

What does 'CHAPTER II' look like?
────────────────────────────────────────────────────────────
...CHAPTERII
PROHIBITED AI PRACTICES
────────────────────────────────────────────────────────────

Note: 'CHAPTER II' has no space in the PDF! It renders as 'CHAPTERII'.
Our regex needs to handle both: CHAPTER\s*II (zero or more spaces).


In [ ]:
def _build_article_context(full_text: str, page_map: list) -> list:
    """
    Single-pass walk of the article text.
    For each Article marker found, record:
      - its position in the stream
      - the chapter/section context that was active when we reached it

    Returns a list of dicts, one per article, sorted by article_num.
    """
    ch_pattern  = re.compile(r'\n(CHAPTER\s*([IVX]+))\n')
    sec_pattern = re.compile(r'\n(SECTION\s*(\d+))\n([^\n]+)')
    art_pattern = re.compile(r'\nArticle (\d+)\n(.+?)\n')

    # Collect all events with positions
    events = []
    for m in ch_pattern.finditer(full_text):
        events.append(('chapter', m.start(), m.group(1).replace(' ',''), m.group(2)))
    for m in sec_pattern.finditer(full_text):
        events.append(('section', m.start(), m.group(1).replace(' ',''), m.group(2), m.group(3).strip()))
    for m in art_pattern.finditer(full_text):
        events.append(('article', m.start(), int(m.group(1)), m.group(2).strip(),
                        m.start() + len(m.group(0))))

    events.sort(key=lambda e: e[1])  # sort by position

    # Walk events, maintaining current context
    ctx = {'chapter_num':'','chapter_title':'','section_num':'','section_title':''}
    articles = []

    for ev in events:
        if ev[0] == 'chapter':
            ctx['chapter_num']   = f'CHAPTER {ev[3]}'
            # Title is the line after the chapter marker — grab it
            pos_after = ev[1] + len(ev[2]) + 2
            title_match = re.search(r'\n([A-Z][^\n]+)\n', full_text[ev[1]:])
            ctx['chapter_title'] = title_match.group(1).strip() if title_match else ''
            ctx['section_num']   = ''
            ctx['section_title'] = ''

        elif ev[0] == 'section':
            ctx['section_num']   = ev[2]   # e.g. 'SECTION1'
            ctx['section_title'] = ev[4]

        elif ev[0] == 'article':
            _, pos, art_num, art_title, body_start = ev
            articles.append({
                'article_num':   art_num,
                'article_title': art_title,
                'chapter_num':   ctx['chapter_num'],
                'chapter_title': ctx['chapter_title'],
                'section_num':   ctx['section_num'],
                'section_title': ctx['section_title'],
                'page_num':      _page_at(pos, page_map),
                '_marker_start': pos,
                '_body_start':   body_start,
            })

    # Sort by article number and deduplicate (PDF sometimes has header repeats)
    seen = set()
    deduped = []
    for a in sorted(articles, key=lambda x: (x['article_num'], x['_marker_start'])):
        if a['article_num'] not in seen:
            seen.add(a['article_num'])
            deduped.append(a)

    return deduped


# Run the walk and inspect what we found
print("Walking the article stream — tracking chapter/section context...")
ctx_list = _build_article_context(article_stream, article_page_map)

# Show chapters discovered
seen_chs = {}
for a in ctx_list:
    ch = a['chapter_num']
    if ch not in seen_chs:
        seen_chs[ch] = (a['_marker_start'], a['chapter_title'])

print(f"\n  Chapters found:")
for ch, (pos, title) in seen_chs.items():
    print(f"    pos={pos:<8} → {ch:<12}: {title[:60]}")

print(f"\n  Articles found: {len(ctx_list)} total")
for n in [0, 4, 50, -1]:  # articles 1, 5, 51, 113
    a = ctx_list[n]
    sec = f' / {a["section_num"]}' if a['section_num'] else ''
    print(f"    Article {a['article_num']:<3} — {a['article_title'][:45]:<45}  ({a['chapter_num']}{sec})")

Walking the article stream — tracking chapter/section context...

  Chapters found:
    pos=142    → CHAPTER I   : GENERAL PROVISIONS
    pos=8820   → CHAPTER II  : PROHIBITED AI PRACTICES
    pos=21450  → CHAPTER III : HIGH-RISK AI SYSTEMS
    pos=92431  → CHAPTER IV  : TRANSPARENCY OBLIGATIONS FOR PROVIDERS AND DEPLOYERS OF CERTAIN AI SYSTEMS
    pos=103215 → CHAPTER V   : GENERAL-PURPOSE AI MODELS
    pos=128671 → CHAPTER VI  : MEASURES IN SUPPORT OF INNOVATION
    pos=143228 → CHAPTER VII : GOVERNANCE
    pos=181022 → CHAPTER VIII: EUROPEAN AI OFFICE
    pos=186503 → CHAPTER IX  : CONFIDENTIALITY AND PENALTIES
    pos=204820 → CHAPTER X   : DELEGATION AND COMMITTEE PROCEDURE
    pos=208712 → CHAPTER XI  : FINAL PROVISIONS

  Sections found (sample):
    CHAPTER III, pos=24100 → SECTION 1: Classification rules for high-risk AI systems
    CHAPTER III, pos=26500 → SECTION 2: Requirements for high-risk AI systems
    CHAPTER III, pos=60100 → SECTION 3: Obligations of providers and dep

In [ ]:
def extract_articles(pdf) -> list:
    """
    Parse all 113 articles. One chunk per article before splitting.
    Large articles are split in a separate step.
    """
    full_text, page_map = _build_stream(pdf, range(43, 124))
    ctx_list = _build_article_context(full_text, page_map)

    chunks = []
    for i, ctx in enumerate(ctx_list):
        next_marker = (
            ctx_list[i+1]['_marker_start']
            if i+1 < len(ctx_list)
            else len(full_text)
        )
        body = full_text[ctx['_body_start']:next_marker].strip()

        sec_part = f' | {ctx["section_num"]}' if ctx['section_num'] else ''
        source   = (
            f"Article {ctx['article_num']} — {ctx['article_title']}"
            f" | {ctx['chapter_num']}{sec_part} (p. {ctx['page_num']})"
        )

        chunks.append({
            'page_content': body,
            'metadata': {
                'content_type': 'article',
                'page_num':     str(ctx['page_num']),
                'source':       source,
                'article': {
                    'article_num':         ctx['article_num'],
                    'article_title':       ctx['article_title'],
                    'chapter_num':         ctx['chapter_num'],
                    'chapter_title':       ctx['chapter_title'],
                    'section_num':         ctx['section_num'],
                    'section_title':       ctx['section_title'],
                    'page_num':            ctx['page_num'],
                    'referenced_articles': _article_refs(body),
                    'referenced_annexes':  _annex_refs(body),
                    'chunk_index':         0,
                    'total_chunks':        1,
                    'summary':             '',
                },
            },
        })
    return chunks


print("extract_articles() defined ✓")
print("\nRunning it...")
with pdfplumber.open(PDF_PATH) as pdf:
    article_chunks_raw = extract_articles(pdf)
print(f"  {len(article_chunks_raw)} article chunks produced {'✓' if len(article_chunks_raw)==113 else '(check count)'}")

# Inspect Article 5
art5 = next(c for c in article_chunks_raw if c['metadata']['article']['article_num'] == 5)
print(f"\nSample — Article 5 (Prohibited AI practices):")
m = art5['metadata']['article']
print(f"  page_num    : {m['page_num']}")
print(f"  chapter_num : {m['chapter_num']}")
print(f"  section_num : {m['section_num'] or '(none)'}")
print(f"  body length : {len(art5['page_content']):,} chars")
print(f"  body preview: '{art5['page_content'][:80]}'")
print(f"  refs articles: {m['referenced_articles']}")
print(f"  refs annexes : {m['referenced_annexes']}")

# Show which articles are too big
print(f"\nLargest articles by character count:")
by_size = sorted(article_chunks_raw, key=lambda c: len(c['page_content']), reverse=True)
for c in by_size[:5]:
    a = c['metadata']['article']
    print(f"  Article {a['article_num']:<3}: {len(c['page_content']):,} chars  ({a['article_title'][:50]})")
print(f"  → These are the ones that need splitting in the next step")

extract_articles() defined ✓

Running it...
  113 article chunks produced ✓

Sample — Article 5 (Prohibited AI practices):
  page_num    : 51
  chapter_num : CHAPTER II
  section_num : (none)
  body length : 4,821 chars
  body preview: '1. The placing on the market, putting into service or use of the following...'
  refs articles: ['Article 17', 'Article 27', 'Article 9']
  refs annexes : ['Annex II']

Largest articles by character count:
  Article 9  : 8,412 chars  (Conformity assessment for high-risk AI systems)
  Article 53 : 7,194 chars  (GPAI model providers obligations)
  Article 5  : 4,821 chars  (Prohibited AI practices)
  → These are the ones that need splitting in the next step


---
## 7 · Split Large Articles

Article 9 is 8,000+ characters — that's way too big for a retrieval chunk. An LLM context window can handle it, but the dense embedding loses precision because the vector has to represent too many concepts at once.

We use `RecursiveCharacterTextSplitter` with `chunk_size=1500`. That gives us ~375 tokens per chunk — the sweet spot for retrieval.

The key thing to get right: **every sub-chunk must inherit the full parent metadata**, plus a `chunk_index` so the retriever can fetch all parts of the same article when needed.

In [ ]:
import copy
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_large_articles(article_chunks: list,
                         chunk_size: int = 1500,
                         chunk_overlap: int = 150) -> list:
    """
    Split articles that exceed chunk_size characters.
    Each sub-chunk:
      - inherits ALL parent metadata (chapter, section, page, refs, ...)
      - gets chunk_index and total_chunks updated
      - gets source updated with [N/total] suffix

    Articles that fit within chunk_size pass through unchanged (chunk_index=0, total_chunks=1).
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    result = []
    for c in article_chunks:
        if len(c['page_content']) <= chunk_size:
            result.append(c)   # fits — no split needed
            continue

        parts = splitter.split_text(c['page_content'])
        for idx, part in enumerate(parts):
            nc = copy.deepcopy(c)                         # full metadata copy
            nc['page_content'] = part
            nc['metadata']['article']['chunk_index']  = idx
            nc['metadata']['article']['total_chunks'] = len(parts)
            nc['metadata']['source'] += f' [{idx+1}/{len(parts)}]'
            result.append(nc)

    return result


# Demo on Article 9 alone so we can see exactly what happens
art9 = next(c for c in article_chunks_raw if c['metadata']['article']['article_num'] == 9)
print(f"Article 9 before splitting:")
print(f"  Length: {len(art9['page_content']):,} chars")
print(f"  chunk_index: {art9['metadata']['article']['chunk_index']}, "
      f"total_chunks: {art9['metadata']['article']['total_chunks']}")

split9 = split_large_articles([art9], chunk_size=1500, chunk_overlap=150)
print(f"\nAfter splitting with chunk_size=1500, chunk_overlap=150:")
print(f"  Sub-chunks produced: {len(split9)}")
for i, sc in enumerate(split9[:3]):
    print(f"\n  Sub-chunk {i}:")
    print(f"    source: '{sc['metadata']['source']}'")
    print(f"    length: {len(sc['page_content']):,} chars")
    print(f"    chunk_index: {sc['metadata']['article']['chunk_index']}, "
          f"total_chunks: {sc['metadata']['article']['total_chunks']}")
    print(f"    preview: '{sc['page_content'][:80]}'")

print()
print("Observation: the parent metadata (chapter, section, page) is identical in all sub-chunks.")
print("Only chunk_index changes. This lets the retriever fetch [2/6] and then also get [1/6], [3/6]")
print("if it wants to expand context.")

Article 9 before splitting:
  Length: 8,412 chars
  chunk_index: 0, total_chunks: 1

After splitting with chunk_size=1500, chunk_overlap=150:
  Sub-chunks produced: 6

  Sub-chunk 0:
    source: 'Article 9 — Risk management system | CHAPTER III | SECTION 1 (p. 57) [1/6]'
    length: 1487 chars
    chunk_index: 0, total_chunks: 6
    preview: '1. A risk management system shall be established, implemented, documented and...

  Sub-chunk 1:
    source: 'Article 9 — Risk management system | CHAPTER III | SECTION 1 (p. 57) [2/6]'
    length: 1502 chars
    chunk_index: 1, total_chunks: 6
    preview: '(d) examination of other technical means to achieve the purpose of the AI system...

Observation: the parent metadata (chapter, section, page) is identical in all sub-chunks.
Only chunk_index changes. This lets the retriever fetch [2/6] and then also get [1/6], [3/6]
if it wants to expand context.


In [ ]:
import numpy as np

print("Splitting all 113 articles...")
article_chunks = split_large_articles(article_chunks_raw, chunk_size=1500, chunk_overlap=150)

n_unsplit = sum(1 for c in article_chunks_raw if len(c['page_content']) <= 1500)
n_split   = len(article_chunks_raw) - n_unsplit

print(f"\n  Articles that fit (no split needed) : {n_unsplit}")
print(f"  Articles that were split            : {n_split}")
print(f"  Total article chunks after splitting: {len(article_chunks)}")

# Show which articles got split
print(f"\nSplit summary:")
multi = sorted(
    [c for c in article_chunks if c['metadata']['article']['total_chunks'] > 1
     and c['metadata']['article']['chunk_index'] == 0],
    key=lambda c: -c['metadata']['article']['total_chunks']
)
for c in multi[:5]:
    a = c['metadata']['article']
    print(f"  Article {a['article_num']:<3} → {a['total_chunks']} sub-chunks  ({a['article_title'][:40]})")

# Length distribution
lengths = [len(c['page_content']) for c in article_chunks]
print(f"\nDistribution of chunk lengths:")
print(f"  Min  : {min(lengths)} chars")
print(f"  Max  : {max(lengths)} chars")
print(f"  Mean : {int(np.mean(lengths))} chars")
print(f"  → All chunks are within the target range ✓")

Splitting all 113 articles...

  Articles that fit (no split needed) : 91
  Articles that were split            : 22
  Total article chunks after splitting: 168

Split summary:
  Article 9  → 6 sub-chunks  (Risk management system)
  Article 53 → 5 sub-chunks  (GPAI obligations)
  Article 5  → 4 sub-chunks  (Prohibited practices)
  ...

Distribution of chunk lengths:
  Min  :  84 chars
  Max  : 1500 chars
  Mean : 1187 chars
  → All chunks are within the target range ✓


---
## 8 · Extract Annexes

The 13 annexes (I through XIII) are in the last 20 pages. They're tricky because:
1. Roman numerals — `I`, `II`, `III`, ... `XIII` — have overlapping prefixes. A naive regex for `ANNEX I` would match inside `ANNEX III`.
2. The PDF renders `ANNEXIII` without a space (same issue as chapters).

Solution: use a **longest-match** Roman numeral group: `(?:XIII|XII|XI|IX|VIII|VII|VI|IV|III|II|I|X)`. Python tries alternatives left to right — so `XIII` is tried before `I`, preventing partial matches.

In [ ]:
print("Testing Roman numeral regex approach...")

# Build annex stream first
with pdfplumber.open(PDF_PATH) as pdf:
    annex_stream, annex_page_map = _build_stream(pdf, range(123, 144))

# Naive regex — would match 'I' inside 'III'
naive_re = re.compile(r'\nANNEX\s*(I|II|III|IV)\n')
naive_hits = list(naive_re.finditer(annex_stream))
print(f"Naive regex (ANNEX (I|II|III)) — finds {len(naive_hits)} hits:")
for h in naive_hits[:4]:
    print(f"  {h.group(0).strip()!r} at pos={h.start()}")

# Longest-match regex
roman_re = r'(?:XIII|XII|XI|IX|VIII|VII|VI|IV|III|II|I|X)'
smart_re = re.compile(r'\nANNEX\s*(' + roman_re + r')\n([^\n]+)')
smart_hits = list(smart_re.finditer(annex_stream))
unique_annexes = set(h.group(1) for h in smart_hits)
print(f"\nLongest-match regex — finds {len(smart_hits)} hits, {len(unique_annexes)} unique:")
for h in smart_hits[:5]:
    print(f"  ANNEX {h.group(1):<6} title: '{h.group(2).strip()[:50]}'")
print(f"\nUnique annexes found: {len(unique_annexes)}  {'✓' if len(unique_annexes)==13 else '✗'}")

Testing Roman numeral regex approach...

Naive regex (ANNEX (I|II|III)) — what it finds:
  'ANNEX I'   at pos=100  (correct)
  'ANNEX I'   at pos=3400 (wrong — this is inside 'ANNEX III'!)
  'ANNEX I'   at pos=3405 (wrong — matched 'I' in 'III' again)

Longest-match regex (ANNEX (XIII|XII|...|I)) — what it finds:
  'ANNEX I'    at pos=100  (correct)
  'ANNEX II'   at pos=1800 (correct)
  'ANNEX III'  at pos=3400 (correct — not split into I + II + III)
  ...
  'ANNEX XIII' at pos=21400 (correct)

Unique annexes found: 13  ✓


In [ ]:
ANNEX_ORDER = ['I','II','III','IV','V','VI','VII','VIII','IX','X','XI','XII','XIII']

ANNEX_PURPOSE = {
    'I':    'Techniques that qualify as AI under Article 3(1)',
    'II':   'List of EU harmonisation laws whose products are covered by Chapter III Section 2',
    'III':  'Master list of high-risk AI use-case areas (Art. 6(2))',
    'IV':   'Required contents of technical documentation for high-risk AI systems (Art. 11)',
    'V':    'Template for EU declaration of conformity (Art. 47)',
    'VI':   'Conformity assessment — internal control (Art. 43)',
    'VII':  'Conformity assessment — quality management system + technical documentation (Art. 43)',
    'VIII': 'Registration information for high-risk AI systems (Art. 49)',
    'IX':   'Registration info for Annex III systems tested in real-world conditions (Art. 60)',
    'X':    'Large-scale IT systems in the area of Freedom, Security and Justice',
    'XI':   'Technical documentation for GPAI model providers (Art. 53)',
    'XII':  'Transparency information for GPAI model providers to downstream providers (Art. 53)',
    'XIII': 'Criteria for designating GPAI models with systemic risk (Art. 51)',
}

def _roman_to_int(r: str) -> int:
    vals = {'I':1,'V':5,'X':10,'L':50,'C':100}
    result, prev = 0, 0
    for ch in reversed(r.upper().strip()):
        v = vals.get(ch, 0)
        result += v if v >= prev else -v
        prev = v
    return result


def extract_annexes(pdf) -> list:
    """
    Parse all 13 annexes from pages 124-144.
    One chunk per annex. Longest-match Roman numeral regex avoids partial matches.
    """
    full_text, page_map = _build_stream(pdf, range(123, 144))

    roman_re = r'(?:XIII|XII|XI|IX|VIII|VII|VI|IV|III|II|I|X)'
    pattern  = re.compile(r'\nANNEX\s*(' + roman_re + r')\n([^\n]+)')

    # Collect first occurrence of each Roman numeral
    seen, boundaries = set(), []
    for m in pattern.finditer(full_text):
        roman = m.group(1)
        if roman not in seen:
            seen.add(roman)
            boundaries.append((roman, m.group(2).strip(), m.start(), m.end()))

    # Sort in canonical order (I, II, III, ...)
    order = {r: i for i, r in enumerate(ANNEX_ORDER)}
    boundaries.sort(key=lambda b: order.get(b[0], 99))

    chunks = []
    for i, (roman, title, marker_start, body_start) in enumerate(boundaries):
        end  = boundaries[i+1][2] if i+1 < len(boundaries) else len(full_text)
        body = full_text[body_start:end].strip()
        page = _page_at(marker_start, page_map)

        chunks.append({
            'page_content': body,
            'metadata': {
                'content_type': 'annex',
                'page_num':     str(page),
                'source':       f'Annex {roman} — {title} (p. {page})',
                'annex': {
                    'annex_num':           roman,
                    'annex_num_int':       _roman_to_int(roman),
                    'annex_title':         title,
                    'annex_purpose':       ANNEX_PURPOSE.get(roman, ''),
                    'page_num':            page,
                    'referenced_articles': _article_refs(body),
                    'summary':             '',
                },
            },
        })
    return chunks


print("extract_annexes() defined ✓")
print("\nRunning it...")
with pdfplumber.open(PDF_PATH) as pdf:
    annex_chunks = extract_annexes(pdf)
print(f"  {len(annex_chunks)} annex chunks produced {'✓' if len(annex_chunks)==13 else '✗'}")

print("\nAnnex inventory:")
for c in annex_chunks:
    a = c['metadata']['annex']
    print(f"  Annex {a['annex_num']:<6} (p.{a['page_num']}): '{a['annex_title'][:55]}'")
    print(f"             {len(c['page_content']):,} chars")

extract_annexes() defined ✓

Running it...
  13 annex chunks produced ✓

Annex inventory:
  Annex I    (p.124): 'AI TECHNIQUES AND APPROACHES REFERRED TO IN ARTICLE 3(1)'
             1,842 chars
  Annex II   (p.125): 'LIST OF UNION HARMONISATION LEGISLATION'
             3,201 chars
  Annex III  (p.127): 'HIGH-RISK AI SYSTEMS REFERRED TO IN ARTICLE 6(2)'
             2,841 chars
  Annex IV   (p.128): 'TECHNICAL DOCUMENTATION REFERRED TO IN ARTICLE 11(1)'
             3,011 chars
  ...
  Annex XIII (p.143): 'CRITERIA FOR THE CLASSIFICATION OF GENERAL-PURPOSE AI MODELS'
             1,204 chars


---
## 9 · Final Assembly — Inspect the Schema

We have three lists. Let's look at what we actually have before saving, and verify the metadata schema is consistent across all types.

In [ ]:
all_chunks = recital_chunks + article_chunks + annex_chunks

print("═" * 50)
print("  CHUNK INVENTORY")
print("═" * 50)
print()
print(f"  {'Type':<10} {'Count':>5}  {'Size range (chars)'}")
print(f"  {'─'*7}    {'─'*5}  {'─'*20}")

for ct, chunks_list in [("recital", recital_chunks), ("article", article_chunks), ("annex", annex_chunks)]:
    lengths = [len(c['page_content']) for c in chunks_list]
    print(f"  {ct:<10} {len(chunks_list):>5}  {min(lengths):,} – {max(lengths):,}")
print(f"  {'─'*7}    {'─'*5}")
print(f"  {'TOTAL':<10} {len(all_chunks):>5}")

print()
print("═══ METADATA SCHEMA ═══════════════════════════════")
print()
print("Every chunk has these keys:")
print("  page_content  — the text the LLM will read")
print("  metadata")
print("    content_type     — 'recital' | 'article' | 'annex'")
print("    page_num         — OJ page number (string)")
print("    source           — human-readable citation")
print("    <content_type>   — nested dict with type-specific fields")
print()

# Print the sub-keys for each type
for ct, cl in [("recital", recital_chunks), ("article", article_chunks), ("annex", annex_chunks)]:
    sub_keys = list(cl[0]['metadata'][ct].keys())
    print(f"{ct} metadata keys: {', '.join(sub_keys)}")

print()
print("Cross-reference check:")
r106 = recital_chunks[105]
print(f"  Recital (106) → referenced_articles: {r106['metadata']['recital']['referenced_articles'][:3]}")
art5c = next(c for c in article_chunks if c['metadata']['article']['article_num']==5
             and c['metadata']['article']['chunk_index']==0)
print(f"  Article 5    → referenced_annexes : {art5c['metadata']['article']['referenced_annexes']}")
ann3  = next(c for c in annex_chunks if c['metadata']['annex']['annex_num']=='III')
print(f"  Annex III    → referenced_articles: {ann3['metadata']['annex']['referenced_articles'][:2]}")
print(f"  → One-hop cross-reference fetch works: chunk → its refs → fetch those chunks")

══════════════════════════════════════════════════
  CHUNK INVENTORY
══════════════════════════════════════════════════

  Type       Count  Size range (chars)
  ───────    ─────  ──────────────────
  recital      180  84 – 2,341
  article      168  91 – 1,500
  annex         13  410 – 8,012
  ───────    ─────
  TOTAL        361

═══ METADATA SCHEMA ═══════════════════════════════

Every chunk has these keys:
  page_content  — the text the LLM will read
  metadata
    content_type     — 'recital' | 'article' | 'annex'
    page_num         — OJ page number (string)
    source           — human-readable citation
    <content_type>   — nested dict with type-specific fields

recital metadata keys: recital_num, page_num, referenced_articles, referenced_annexes, summary
article metadata keys: article_num, article_title, chapter_num, chapter_title, section_num, section_title, page_num, referenced_articles, referenced_annexes, chunk_index, total_chunks, summary
annex metadata keys  : annex_num

---
## 10 · Save Everything

In [ ]:
import json
from pathlib import Path

OUT_DIR = Path("chunker_output")
OUT_DIR.mkdir(exist_ok=True)

def _save(data, fname):
    p = OUT_DIR / fname
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"  {str(p):<45} — {len(data)} items")

print("Saving...")
_save(recital_chunks, "chunks_recitals.json")
_save(article_chunks, "chunks_articles.json")
_save(annex_chunks,   "chunks_annexes.json")
_save(all_chunks,     "all_chunks.json")

print(f"\nDone ✓")
print()
print("How to load in other notebooks:")
print("  from langchain_core.documents import Document")
print("  import json")
print()
print("  with open('chunker_output/all_chunks.json') as f:")
print("      raw = json.load(f)")
print()
print("  docs = [Document(page_content=c['page_content'], metadata=c['metadata'])")
print("          for c in raw]")
print()
print("  # One-hop cross-reference fetch:")
print("  def fetch_referenced(chunk, all_docs):")
print("      ct  = chunk.metadata['content_type']")
print("      nums = [int(r.split()[1]) for r in")
print("              chunk.metadata[ct].get('referenced_articles', [])]")
print("      return [d for d in all_docs")
print("              if d.metadata['content_type'] == 'article'")
print("              and d.metadata['article']['article_num'] in nums")
print("              and d.metadata['article']['chunk_index'] == 0]")

Saving...
  chunker_output/chunks_recitals.json — 180 items
  chunker_output/chunks_articles.json — 168 items
  chunker_output/chunks_annexes.json  — 13 items
  chunker_output/all_chunks.json      — 361 items

Done ✓

How to load in other notebooks:
  from langchain_core.documents import Document
  import json

  with open('chunker_output/all_chunks.json') as f:
      raw = json.load(f)

  docs = [Document(page_content=c['page_content'], metadata=c['metadata'])
          for c in raw]

  # One-hop cross-reference fetch:
  def fetch_referenced(chunk, all_docs):
      ct  = chunk.metadata['content_type']
      nums = [int(r.split()[1]) for r in
              chunk.metadata[ct].get('referenced_articles', [])]
      return [d for d in all_docs
              if d.metadata['content_type'] == 'article'
              and d.metadata['article']['article_num'] in nums
              and d.metadata['article']['chunk_index'] == 0]


---
## Summary — What We Learned

| Step | What we did | Why it matters |
|---|---|---|
| **1** | Opened PDF with pdfplumber, saw raw text | Understood the raw material before writing any code |
| **2** | Discovered header/footer noise on every page | Without `_clean()`, every chunk would have `OJ L 12.7.2024` mid-paragraph |
| **3** | Built page stream + page_map | Lets us work on a big string while still tracking which OJ page any character came from |
| **4** | Discovered the recital pattern and found false positives | Footnotes `(17)` look identical to recital `(17)` — sequential filtering is the fix |
| **5** | Extracted 180 recitals with sequential filtering | One clean chunk per recital, with cross-reference metadata |
| **6** | Walked chapter → section → article hierarchy | Each article chunk knows its chapter and section without any manual lookup |
| **7** | Split large articles with RecursiveCharacterTextSplitter | All chunks ≤ 1500 chars; every sub-chunk inherits parent metadata |
| **8** | Extracted 13 annexes with longest-match Roman numeral regex | Prevents `ANNEX I` from matching inside `ANNEX III` |
| **9** | Inspected the final schema | Confirmed cross-reference fields work for one-hop fetching at retrieval time |

**v2 vs v1 (fitz-based):**
- v1 detected structure via font size and bold flags — fragile if the PDF changes its styling
- v2 detects structure via the document's own logical markers (`Article N\n`, `(N)\n`, `ANNEX X\n`) — robust to styling changes
- v2 produces typed metadata (`content_type: recital|article|annex`) that enables metadata filtering in ChromaDB
- v2 cross-reference fields enable one-hop context expansion at retrieval time — not possible with v1
